In [ ]:
# GE2E game-embedding fingerprint, WITH a timing channel (ablation vs moves-only).
# Groups the cached per-ply features into games, encodes each game -> one style vector,
# trains with GE2E (speaker-verification loss) so vectors cluster by player. Flip USE_TIME
# to answer: does the clock channel help GE2E identification?
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-chess", "zstandard"])
import os, glob, math, random, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); random.seed(0); np.random.seed(0)

USE_TIME = True        # <-- the ablation switch. Run once True, once False, compare.
D_MODEL  = 256         # game-encoder width
N_LAYERS = 4
N_HEADS  = 4
MAX_PLIES = 60         # cap plies per game (the player's own moves)
M_PLAYERS = 32         # GE2E batch: players per step
K_GAMES   = 5          # GE2E batch: games per player per step
STEPS     = 3000
LR        = 3e-4

CACHE = "/kaggle/working/cache_film.pt"
_hits = [CACHE] if os.path.exists(CACHE) else glob.glob("/kaggle/input/**/cache_film.pt", recursive=True)
assert _hits, "cache_film.pt not found — attach your cache dataset."
print("loading", _hits[0]); c = torch.load(_hits[0])
pooled = c["pooled"].float(); think = c["think"].float()
game = c["game"].long(); pid = c["pid"].long(); test = c["test"].bool()
names = c["names"]; P = len(names); Dp = pooled.shape[1]
print(f"{pooled.shape[0]} plies | {P} players | pooled dim {Dp} | mode: {'MOVES+TIME' if USE_TIME else 'MOVES-ONLY'}")

In [ ]:
# ---- group plies -> games (cache keeps a game's plies contiguous & in order) ----
order = torch.argsort(game, stable=True)
gs = game[order]; po = pooled[order]; th = think[order]; pi = pid[order]; te = test[order]
change = torch.ones(len(gs), dtype=torch.bool); change[1:] = gs[1:] != gs[:-1]
starts = change.nonzero(as_tuple=True)[0].tolist() + [len(gs)]
G = len(starts) - 1

# per-ply timing feature: standardized log think-time
logt = torch.log1p(th.clamp(min=0))
tmean, tstd = logt.mean(), logt.std().clamp(min=1e-3)

gpool = torch.zeros(G, MAX_PLIES, Dp, dtype=torch.float16)
gtime = torch.zeros(G, MAX_PLIES)
gmask = torch.zeros(G, MAX_PLIES, dtype=torch.bool)
gown  = torch.zeros(G, dtype=torch.long)
gtest = torch.zeros(G, dtype=torch.bool)
for i in range(G):
    a, b = starts[i], starts[i + 1]
    l = min(b - a, MAX_PLIES)
    gpool[i, :l] = po[a:a + l].half()
    gtime[i, :l] = (logt[order][a:a + l] - tmean) / tstd
    gmask[i, :l] = True
    gown[i] = pi[a]; gtest[i] = te[a]

gpool = gpool.to(DEV); gtime = gtime.to(DEV); gmask = gmask.to(DEV)
gown = gown.to(DEV); gtest = gtest.to(DEV)
del pooled, think, po, th, order, logt
print(f"{G} games | {int(gtest.sum())} test | avg plies {gmask.float().sum(1).mean():.1f}")

# reference (enrollment/training) games grouped by player, for GE2E sampling
train_by_player = {}
for i in range(G):
    if not bool(gtest[i]):
        train_by_player.setdefault(int(gown[i]), []).append(i)
elig = [p for p, gl in train_by_player.items() if len(gl) >= K_GAMES]
print(f"{len(elig)} players have >= {K_GAMES} reference games (usable for GE2E batches)")

In [ ]:
class GameEncoder(nn.Module):
    """Per-ply [pooled (+ timing)] -> transformer over the game -> one L2-normalized vector."""
    def __init__(self, dp, d=D_MODEL, use_time=USE_TIME, layers=N_LAYERS, heads=N_HEADS, L=MAX_PLIES):
        super().__init__()
        self.use_time = use_time
        in_dim = dp + (1 if use_time else 0)
        self.proj = nn.Linear(in_dim, d)
        self.pos = nn.Parameter(torch.zeros(1, L, d))
        enc = nn.TransformerEncoderLayer(d, heads, d * 4, dropout=0.1,
                                         batch_first=True, activation="gelu")
        self.tr = nn.TransformerEncoder(enc, layers)
    def forward(self, pool, tfeat, mask):
        x = pool.float()
        if self.use_time:
            x = torch.cat([x, tfeat.unsqueeze(-1)], dim=-1)
        x = self.proj(x) + self.pos[:, :x.shape[1]]
        h = self.tr(x, src_key_padding_mask=~mask)         # ignore padded plies
        m = mask.unsqueeze(-1).float()
        pooledg = (h * m).sum(1) / m.sum(1).clamp(min=1)   # masked mean
        return F.normalize(pooledg, dim=-1)

class GE2E(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(10.0)); self.b = nn.Parameter(torch.tensor(-5.0))
    def loss(self, emb):                                    # emb [M,K,d], L2-normalized
        M, K, d = emb.shape
        cent = F.normalize(emb.mean(1), dim=-1)             # [M,d] player centroids
        excl = F.normalize((emb.sum(1, keepdim=True) - emb) / (K - 1), dim=-1)  # leave-one-out
        sim = torch.einsum("mkd,jd->mkj", emb, cent)        # cosine to every centroid
        own = (emb * excl).sum(-1)                          # own uses leave-one-out centroid
        ar = torch.arange(M, device=emb.device)
        sim[ar, :, ar] = own
        sim = self.w.clamp(min=1e-6) * sim + self.b
        tgt = ar[:, None].expand(M, K).reshape(-1)
        return F.cross_entropy(sim.reshape(M * K, M), tgt)

enc = GameEncoder(Dp).to(DEV)
ge2e = GE2E().to(DEV)
print(f"encoder params: {sum(p.numel() for p in enc.parameters())/1e6:.2f}M")

In [ ]:
opt = torch.optim.AdamW(list(enc.parameters()) + list(ge2e.parameters()), lr=LR, weight_decay=1e-4)
rng = random.Random(0)
enc.train(); t0 = time.time()
for step in range(STEPS):
    players = rng.sample(elig, M_PLAYERS)
    idx = []
    for p in players:
        idx += rng.sample(train_by_player[p], K_GAMES)
    idx = torch.tensor(idx, device=DEV)
    e = enc(gpool[idx], gtime[idx], gmask[idx]).view(M_PLAYERS, K_GAMES, -1)
    loss = ge2e.loss(e)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(enc.parameters(), 3.0)
    opt.step()
    if (step + 1) % 250 == 0:
        print(f"step {step+1}/{STEPS}  ge2e loss {loss.item():.4f}  ({time.time()-t0:.0f}s)")

In [ ]:
@torch.no_grad()
def embed_all():
    enc.eval(); E = torch.zeros(G, D_MODEL, device=DEV); B = 512
    for s in range(0, G, B):
        E[s:s+B] = enc(gpool[s:s+B], gtime[s:s+B], gmask[s:s+B])
    return E

E = embed_all()
tr = ~gtest
# enroll: player centroid from their reference-game embeddings
cent = torch.zeros(P, D_MODEL, device=DEV)
for p in range(P):
    m = tr & (gown == p)
    if m.any(): cent[p] = F.normalize(E[m].mean(0), dim=-1)
cent = F.normalize(cent, dim=-1)

test_idx = gtest.nonzero(as_tuple=True)[0]
Et = E[test_idx]; own_t = gown[test_idx]

def p_at_1(Nsub):
    keep = own_t < Nsub
    sim = Et[keep] @ cent[:Nsub].T
    return (sim.argmax(1) == own_t[keep]).float().mean().item()

print(f"\n=== GE2E {'MOVES+TIME' if USE_TIME else 'MOVES-ONLY'} — single game / query ===")
print("players(N)   P@1     chance")
for Nsub in [10, 30, 60, P]:
    print(f"{Nsub:>8}   {p_at_1(Nsub):.3f}   {1/Nsub:.3f}")

# games-per-query: average k test-game embeddings of one player, then nearest centroid over all P
by = {}
for j in range(len(test_idx)):
    by.setdefault(int(own_t[j]), []).append(j)
def games_per_query(ks=(1,3,5,10), trials=400):
    r = random.Random(0)
    owners = [p for p, l in by.items() if len(l) >= max(ks)]
    print(f"\n=== GE2E {'MOVES+TIME' if USE_TIME else 'MOVES-ONLY'} — games/query (N={P}) ===")
    print("games   P@1")
    for k in ks:
        hit = tot = 0
        for _ in range(trials):
            p = r.choice(owners); sel = r.sample(by[p], k)
            q = F.normalize(Et[sel].mean(0), dim=-1)
            hit += int((q @ cent.T).argmax().item() == p); tot += 1
        print(f"{k:>5}   {hit/tot:.3f}")
games_per_query()